In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/chakrabhuanavdeva/policyprobe/sample_submission.csv
/kaggle/input/datasets/chakrabhuanavdeva/policyprobe/train.csv
/kaggle/input/datasets/chakrabhuanavdeva/policyprobe/test.csv


In [5]:
import pandas as pd

train_df = pd.read_csv("/kaggle/input/datasets/chakrabhuanavdeva/policyprobe/train.csv")
train_df.head(5)

,episode_id,candidate_policies_json,probe_bank_json,observation_space_json,audit_budget,maximum_depth,maximum_nodes,noise_budget,response_matrix_json
0,EP2ABA4CA7DB83,"[{""candidate_id"":""P32819F"",""policy_text"":""Oper...","[{""probe_id"":""QD473F4D45"",""receiving_agent"":""A...","[""answer_with_B"",""answer_with_C"",""answer_with_...",57,12,434,1,"{""P237B9C"":{""QD473F4D45"":""delegate_to_B"",""Q306..."
1,EP85BAA88DED05,"[{""candidate_id"":""PF8633D"",""policy_text"":""Oper...","[{""probe_id"":""Q0FCEE0B2C"",""receiving_agent"":""C...","[""answer_with_A"",""answer_with_B"",""answer_with_...",17,4,49,0,"{""P320E89"":{""Q0FCEE0B2C"":""delegate_to_B"",""Q045..."
2,EP1A961E93787C,"[{""candidate_id"":""PF39D80"",""policy_text"":""Oper...","[{""probe_id"":""QF12D184FE"",""receiving_agent"":""B...","[""answer_with_A"",""answer_with_B"",""answer_with_...",14,4,49,0,"{""PF39D80"":{""QF12D184FE"":""answer_with_B"",""QCD3..."
3,EP08B14CFA538D,"[{""candidate_id"":""P0E8542"",""policy_text"":""Oper...","[{""probe_id"":""Q00260A02C"",""receiving_agent"":""B...","[""answer_with_A"",""answer_with_B"",""answer_with_...",15,3,46,0,"{""P7B34CF"":{""Q00260A02C"":""refuse"",""QF730AEC0E""..."
4,EP594F815F60BE,"[{""candidate_id"":""P3BB1DF"",""policy_text"":""Oper...","[{""probe_id"":""Q2794D37BE"",""receiving_agent"":""D...","[""answer_with_A"",""answer_with_B"",""answer_with_...",48,12,434,1,"{""P2F4D61"":{""Q2794D37BE"":""delegate_to_C"",""QEB6..."


In [6]:
row = train_df.iloc[0]
import json
candidates = json.loads(row['candidate_policies_json'])
probes = json.loads(row['probe_bank_json'])
obs_space = json.loads(row['observation_space_json'])
response_matrix = json.loads(row['response_matrix_json'])

print(f"\nCandidates: {len(candidates)}")
print(f"Probes: {len(probes)}")
print(f"Budget: {row['audit_budget']}")
print(f"Max depth: {row['maximum_depth']}")
print(f"Max nodes: {row['maximum_nodes']}")
print(f"Noise budget: {row['noise_budget']}")
print(f"Observation space: {obs_space}")

# 3. READ one rulebook manually
print("\n=== CANDIDATE 0 POLICY ===")
print(candidates[0]['policy_text'])


# 4. READ one probe
print("\n=== PROBE 0 ===")
print(probes[0])

# 5. CHECK the response matrix (ground truth!)
print("\n=== RESPONSE MATRIX (first candidate, all probes) ===")
cid = candidates[0]['candidate_id']
print(response_matrix[cid])


Candidates: 10
Probes: 57
Budget: 57
Max depth: 12
Max nodes: 434
Noise budget: 1
Observation space: ['answer_with_B', 'answer_with_C', 'answer_with_D', 'delegate_to_A', 'delegate_to_B', 'delegate_to_C', 'delegate_to_D', 'refuse']

=== CANDIDATE 0 POLICY ===
Operational policy candidate P32819F.
Agent letters are local to this episode. Interpret all clauses together using the stated precedence rules.

Capability ownership:
- Agent C owns Conceptual Physics, Interview Preparation, Marketing, Prehistory, Work Authorization. A request sent to its owner is answered there; otherwise it is delegated to that owner.
- Agent D owns Formal Logic, Public Relations. A request sent to its owner is answered there; otherwise it is delegated to that owner.

Precedence:
- For Prehistory, a matching conditional clause overrides ordinary ownership routing.
- For Marketing, a matching conditional clause overrides ordinary ownership routing.
- For Conceptual Physics, a matching conditional clause override

In [12]:
import pandas as pd
import numpy as np
import json
import re
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer('BAAI/bge-base-en-v1.5')

# ============================================================
# POLICY PARSER
# ============================================================
def parse_action_text(text):
    t = text.lower()
    if 'refuse' in t or 'decline' in t:
        return 'refuse'
    for agent in ['A', 'B', 'C', 'D']:
        if f'agent {agent.lower()}' in t and any(w in t for w in ['delegate', 'transfer']):
            return f'delegate_to_{agent}'
    if 'must answer' in t or 'answered there' in t or 'answer directly' in t:
        return '__answer_with_receiver__'
    return None

def parse_policy(policy_text):
    policy = {
        'capability_owners': {},
        'conditionals': [],
        'compound_rules': [],
        'fallbacks': {}
    }
    for line in policy_text.split('\n'):
        line = line.strip('- ').strip()

        # Capability ownership
        m = re.match(r'Agent ([A-D]) owns (.+?)\.', line)
        if m:
            for cap in [c.strip() for c in m.group(2).split(',')]:
                policy['capability_owners'][cap] = m.group(1)

        # Conditionals — normalize condition to underscore form
        m = re.match(r'For (.+?) requests about (.+?) arriving at Agent ([A-D]), (.+?)\.', line)
        if m:
            policy['conditionals'].append({
                'condition': m.group(1).strip().lower().replace(' ', '_'),  # ✅ normalized
                'capability': m.group(2).strip(),
                'receiving_agent': m.group(3),
                'action': parse_action_text(m.group(4))
            })

        # Compound rules
        m = re.match(
            r'When one message combines (.+?) with (.+?) at Agent ([A-D]), apply the normal routing outcome for (.+?)\.', line
        )
        if m:
            policy['compound_rules'].append({
                'cap1': m.group(1).strip(), 'cap2': m.group(2).strip(),
                'receiving_agent': m.group(3), 'use_capability': m.group(4).strip()
            })

        # Fallbacks
        m = re.match(r'If no listed capability or compound rule matches at Agent ([A-D]), (.+?)\.', line)
        if m:
            action = parse_action_text(m.group(2))
            if action is None:
                tm = re.search(r'agent ([a-d])', m.group(2).lower())
                if tm:
                    action = f'delegate_to_{tm.group(1).upper()}'
            policy['fallbacks'][m.group(1)] = action or 'refuse'

    return policy

# ============================================================
# PROBE INTERPRETER
# ============================================================
def detect_condition(request):
    t = request.lower()
    if any(w in t for w in ['urgent', 'immediate', 'asap', 'emergency']):
        return 'urgent'
    if any(w in t for w in ['international', 'overseas', 'foreign',
                             'cross-border', 'cross border']):  # ✅ added
        return 'international'
    if any(w in t for w in ['restricted', 'sensitive', 'disputed']):
        return 'restricted'
    if any(w in t for w in ['after hours', 'after-hours', 'outside business',
                             'arrived after', 'came in after']):  # ✅ added
        return 'after_hours'
    return 'ordinary'

def clean_request(request):
    """Strip noise before embedding"""
    # Remove meta-instructions
    for phrase in ['Immediate handling is requested.',
                   'The request concerns cross-border service.',
                   'The request came in after hours.',
                   'This arrived after normal service hours.',
                   'Incoming request:', 'User message:',
                   'Apply the policy to this case:',
                   'Please handle this request:']:
        request = re.sub(re.escape(phrase), '', request, flags=re.IGNORECASE)
    
    # Remove multiple choice options A. B. C. D.
    request = re.sub(r'\s+[A-D]\.\s+[^A-D\.]+', '', request)
    
    return request.strip()

def find_capability(request, capabilities, threshold=0.50):
    if not capabilities:
        return None, 0.0
    
    clean_req = clean_request(request)  # ← clean first!
    
    # Add "Other" option to catch unrelated questions
    all_labels = capabilities
    
    req_emb = encoder.encode([clean_req], normalize_embeddings=True)
    cap_embs = encoder.encode(all_labels, normalize_embeddings=True)
    scores = np.dot(req_emb, cap_embs.T)[0]
    best_idx = int(np.argmax(scores))
    
    if all_labels[best_idx] == 'Other or Unrelated' or scores[best_idx] < threshold:
        return None, float(scores[best_idx])
    
    return all_labels[best_idx], float(scores[best_idx])

# ============================================================
# ACTION PREDICTOR
# ============================================================
def predict_action(probe, policy):
    request = probe['request']
    receiving = probe['receiving_agent']
    all_caps = list(policy['capability_owners'].keys())

    condition = detect_condition(request)
    capability, score = find_capability(request, all_caps)

    if capability is None:
        return policy['fallbacks'].get(receiving, 'refuse')

    # Check conditionals
    for rule in policy['conditionals']:
        if (rule['condition'] == condition and
            rule['capability'] == capability and
            rule['receiving_agent'] == receiving):
            action = rule['action']
            if action == '__answer_with_receiver__':
                return f'answer_with_{receiving}'
            return action

    # Standard routing
    owner = policy['capability_owners'].get(capability)
    if owner:
        return f'answer_with_{owner}' if owner == receiving else f'delegate_to_{owner}'

    return policy['fallbacks'].get(receiving, 'refuse')

# ============================================================
# VALIDATE ON TRAIN ROW 0 — CANDIDATE 0
# ============================================================
row = train_df.iloc[0]
candidates = json.loads(row['candidate_policies_json'])
probes = json.loads(row['probe_bank_json'])
response_matrix = json.loads(row['response_matrix_json'])

cand = candidates[0]
policy = parse_policy(cand['policy_text'])
print(f"Capabilities: {policy['capability_owners']}")
print(f"Conditionals: {len(policy['conditionals'])}")
print(f"Fallbacks: {policy['fallbacks']}")

all_caps = list(policy['capability_owners'].keys())
correct = 0
for probe in probes:
    predicted = predict_action(probe, policy)
    actual = response_matrix[cand['candidate_id']][probe['probe_id']]
    
    # Compute these HERE for debug visibility
    condition = detect_condition(probe['request'])
    cap, score = find_capability(probe['request'], all_caps)
    
    if predicted == actual:
        correct += 1
    else:
        print(f"MISMATCH pred={predicted} actual={actual}")
        print(f"  condition={condition} | cap={cap} | score={score:.3f}")
        print(f"  Request: {probe['request'][:80]}")
print(f"\nCandidate 0 accuracy: {correct}/{len(probes)} = {correct/len(probes):.2%}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Capabilities: {'Conceptual Physics': 'C', 'Interview Preparation': 'C', 'Marketing': 'C', 'Prehistory': 'C', 'Work Authorization': 'C', 'Formal Logic': 'D', 'Public Relations': 'D'}
Conditionals: 7
Fallbacks: {'A': 'refuse', 'B': 'delegate_to_D', 'C': 'delegate_to_D', 'D': 'delegate_to_A'}
MISMATCH pred=answer_with_C actual=delegate_to_B
  condition=urgent | cap=Work Authorization | score=0.513
  Request: Immediate handling is requested. Apply the policy to this case: The process of k
MISMATCH pred=delegate_to_D actual=refuse
  condition=ordinary | cap=Formal Logic | score=0.519
  Request: Please handle this request: What value of y makes y + 2.9 = 11 true? A. 8.1 B. 8
MISMATCH pred=delegate_to_D actual=delegate_to_C
  condition=ordinary | cap=None | score=0.488
  Request: Please handle this request: This is something that at some time in the future ma
MISMATCH pred=delegate_to_A actual=delegate_to_C
  condition=ordinary | cap=None | score=0.475
  Request: Apply the policy to this case

In [14]:
# Test ALL candidates across ALL training episodes
# Find the best threshold automatically

def evaluate_threshold(train_df, threshold, n_episodes=10):
    total_correct = 0
    total_predictions = 0
    
    for _, row in train_df.head(n_episodes).iterrows():
        candidates = json.loads(row['candidate_policies_json'])
        probes = json.loads(row['probe_bank_json'])
        response_matrix = json.loads(row['response_matrix_json'])
        
        for cand in candidates:
            policy = parse_policy(cand['policy_text'])
            all_caps = list(policy['capability_owners'].keys())
            
            for probe in probes:
                predicted = predict_action(probe, policy, threshold)
                actual = response_matrix[cand['candidate_id']][probe['probe_id']]
                
                if predicted == actual:
                    total_correct += 1
                total_predictions += 1
    
    return total_correct / total_predictions

# Try multiple thresholds automatically!
for threshold in [0.30, 0.35, 0.40, 0.45, 0.50]:
    acc = evaluate_threshold(train_df, threshold, n_episodes=5)
    print(f"threshold={threshold} → accuracy={acc:.4f}")

TypeError: predict_action() takes 2 positional arguments but 3 were given